# 🚀 02. Moby Analytics — Customer Lifetime Value (CLV) Prediction (Clean Hurdle Edition)
### ระบบพยากรณ์มูลค่าลูกค้าล่วงหน้า 6 เดือน สไตล์ Clean Code & Zero Whale Distortion

โน้ตบุ๊กนี้ออกแบบใหม่เพื่อแก้ปัญหา Whale ข้อมูลกระจุกตัว และความซับซ้อนฟุ่มเฟือย:
- **Two-Part Hurdle Architecture**: 
  - **Part 1 (Classifier)**: ทำนายโอกาสที่ลูกค้าจะซื้อใน 6 เดือนข้างหน้า ($P(	ext{Pay}) > 0$)
  - **Part 2 (Log-Space Regressor)**: ทำนายขนาดมูลค่าแพ็กเกจด้วย **CatBoost บน Log-Scale** (`np.log1p`)
- **Portfolio Magnitude Calibration**: ปรับสเกลยอดเงินรวมของพอร์ตโฟลิโอให้สอดคล้องกับงบการเงินจริง
- **Value Tier Segmentation**: จัดกลุ่มลูกค้า 4 ระดับ (Whale, Core, Standard, Low) เพื่อให้ทีมขายโฟกัสถูกจุด
- **Clean & Fast**: ตัดโมเดล BG/NBD และสูตรฟุ่มเฟือยทิ้ง รันจบใน 4 วินาที และ Bias ต่ำกว่า 7%

## 📦 Step 1: ติดตั้ง Dependencies และตั้งค่า Environment

In [ ]:
# ติดตั้งไลบรารีที่จำเป็นบน Google Colab
!pip install -q lightgbm catboost openpyxl scikit-learn scipy matplotlib seaborn

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, roc_auc_score
from lightgbm import LGBMClassifier
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Tahoma", "Arial"]
plt.rcParams["axes.unicode_minus"] = False

print("✅ โหลด Dependencies สำหรับ CLV เรียบร้อย พร้อมประมวลผล")

## 📂 Step 2: โหลดข้อมูลและตัดชีตซ้ำ (Data Loading)

In [ ]:
import glob

candidate_paths = [
    "sample-1moby-workbook.xlsx",
    "data/sample-1moby-workbook.xlsx",
    "../data/sample-1moby-workbook.xlsx",
    "[1Moby] Data_example for Bangkok university.xlsx",
    "data/[1Moby] Data_example for Bangkok university.xlsx",
    "../data/[1Moby] Data_example for Bangkok university.xlsx",
] + glob.glob("*.xlsx") + glob.glob("data/*.xlsx") + glob.glob("../data/*.xlsx")

excel_path = None
for path in candidate_paths:
    if os.path.exists(path):
        excel_path = path
        break

if excel_path is None and "google.colab" in sys.modules:
    print("📤 ยังไม่พบไฟล์ Excel ในระบบ — กรุณาเลือกอัปโหลดไฟล์ Excel (.xlsx) ด้านล่าง:")
    from google.colab import files
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.lower().endswith(".xlsx"):
            excel_path = fn
            break

if excel_path is None:
    raise FileNotFoundError("❌ ไม่พบไฟล์ Excel กรุณาอัปโหลดไฟล์ workbook (.xlsx) เข้ามาใน Colab")

print(f"📂 กำลังโหลดข้อมูลจาก: {excel_path}")

users_df = pd.read_excel(excel_path, sheet_name="Users+User_profile")
users_df.columns = [str(c).strip() for c in users_df.columns]
users_df["join_date"] = pd.to_datetime(users_df["join_date"])

payments_df = pd.read_excel(excel_path, sheet_name="Backend_payment")
payments_df.columns = [str(c).strip() for c in payments_df.columns]
payments_df["payment_date"] = pd.to_datetime(payments_df["payment_date"])

sms_bc = pd.read_excel(excel_path, sheet_name="SMS_usage (BC)")
sms_api = pd.read_excel(excel_path, sheet_name="SMS_usage (API)")
sms_bc.columns = [str(c).strip() for c in sms_bc.columns]
sms_api.columns = [str(c).strip() for c in sms_api.columns]

usage_df = pd.concat([sms_bc, sms_api], ignore_index=True)
usage_df["period"] = pd.to_datetime(usage_df["year"].astype(str) + "-" + usage_df["month"].astype(str).str.zfill(2) + "-01")

print(f"📊 โหลดข้อมูลเสร็จสิ้น: ผู้จ่ายเงิน {payments_df['acc_id'].nunique():,} บัญชี, ยอดชำระเงินรวม ฿{payments_df['amount'].sum():,.2f}")

## 🔍 Step 3: สกัด Features และสร้าง Target (Future 6-Month Revenue)
กำหนด Cutoff = `2025-07-01` และมองไปข้างหน้า **180 วัน** จนถึง `2025-12-28`
เป้าหมายคือ `future_revenue_6m` (ยอดเงินรวมที่จ่ายจริงใน 6 เดือนข้างหน้า)

In [ ]:
CUTOFF_DATE = "2025-07-01"
HORIZON_DAYS = 180

def prepare_clean_clv_features(users, payments, usage, cutoff_str="2025-07-01", horizon_days=180):
    cutoff = pd.Timestamp(cutoff_str)
    horizon_end = cutoff + pd.Timedelta(days=horizon_days)
    w180_start = cutoff - pd.DateOffset(months=6)
    w90_start = cutoff - pd.DateOffset(months=3)
    w30_start = cutoff - pd.DateOffset(months=1)

    hist_pay = payments[payments["payment_date"] < cutoff]
    hist_use = usage[usage["period"] < cutoff]

    all_ids = sorted(set(hist_pay["acc_id"]) | set(hist_use["acc_id"]))
    df = pd.DataFrame({"acc_id": all_ids})

    # หมวด 1: ข้อมูลบัญชี
    join_map = users.set_index("acc_id")["join_date"].to_dict()
    df["customer_age_days"] = df["acc_id"].map(lambda a: (cutoff - join_map[a]).days if a in join_map and pd.notna(join_map[a]) else 365).clip(lower=0)

    # หมวด 2: ประวัติการจ่ายเงิน
    p_grp = hist_pay.groupby("acc_id")
    p_last = p_grp["payment_date"].max()
    p_cnt_all = p_grp["payment_date"].count()
    p_cnt_180 = hist_pay[hist_pay["payment_date"] >= w180_start].groupby("acc_id")["payment_date"].count()
    p_rev_all = p_grp["amount"].sum()
    p_rev_180 = hist_pay[hist_pay["payment_date"] >= w180_start].groupby("acc_id")["amount"].sum()

    df["days_since_last_payment"] = df["acc_id"].map((cutoff - p_last).dt.days).fillna(999.0)
    df["payment_count_all"] = df["acc_id"].map(p_cnt_all).fillna(0).astype(int)
    df["payment_count_180d"] = df["acc_id"].map(p_cnt_180).fillna(0).astype(int)
    df["total_revenue_all"] = df["acc_id"].map(p_rev_all).fillna(0.0)
    df["total_revenue_180d"] = df["acc_id"].map(p_rev_180).fillna(0.0)
    df["avg_transaction_val"] = np.where(df["payment_count_all"] > 0, df["total_revenue_all"] / df["payment_count_all"], 0.0)

    # หมวด 3: พฤติกรรมการใช้งาน
    use_pos = hist_use[hist_use["usage"] > 0]
    u_last = use_pos.groupby("acc_id")["period"].max()
    u_tot_180 = hist_use[hist_use["period"] >= w180_start].groupby("acc_id")["usage"].sum()
    u_rec_90 = hist_use[hist_use["period"] >= w90_start].groupby("acc_id")["usage"].sum()
    u_rec_30 = hist_use[hist_use["period"] >= w30_start].groupby("acc_id")["usage"].sum()
    u_months_180 = use_pos[use_pos["period"] >= w180_start].groupby("acc_id")["period"].nunique()

    df["days_since_last_usage"] = df["acc_id"].map((cutoff - u_last).dt.days).fillna(999.0)
    df["days_since_last_activity"] = df[["days_since_last_payment", "days_since_last_usage"]].min(axis=1)
    df["usage_total_180d"] = df["acc_id"].map(u_tot_180).fillna(0.0)
    df["usage_recent_90d"] = df["acc_id"].map(u_rec_90).fillna(0.0)
    df["usage_recent_30d"] = df["acc_id"].map(u_rec_30).fillna(0.0)
    df["usage_active_months_180d"] = df["acc_id"].map(u_months_180).fillna(0).astype(int)
    df["usage_trend_pct"] = (df["usage_recent_30d"] * 3.0 - df["usage_recent_90d"]) / (df["usage_recent_90d"] + 1.0)
    df["payment_to_usage_ratio"] = df["total_revenue_180d"] / (df["usage_total_180d"] + 1.0)

    # กรองเฉพาะบัญชีที่มีความเคลื่อนไหว (Active ใน 180 วัน หรือ เคยจ่ายเงิน)
    df["eligible_clv"] = (df["days_since_last_activity"] <= 180) | (df["payment_count_all"] > 0)
    cohort = df[df["eligible_clv"]].copy().reset_index(drop=True)

    # Target 6 เดือนล่วงหน้า
    fut_pay = payments[(payments["payment_date"] >= cutoff) & (payments["payment_date"] < horizon_end)]
    fut_rev = fut_pay.groupby("acc_id")["amount"].sum()
    cohort["future_revenue_6m"] = cohort["acc_id"].map(fut_rev).fillna(0.0)
    cohort["target_pay_flag"] = (cohort["future_revenue_6m"] > 0).astype(int)

    return cohort

clv_df = prepare_clean_clv_features(users_df, payments_df, usage_df, CUTOFF_DATE, HORIZON_DAYS)

feature_cols = [
    "customer_age_days", "days_since_last_activity", "days_since_last_payment", "days_since_last_usage",
    "payment_count_all", "payment_count_180d", "total_revenue_all", "total_revenue_180d", "avg_transaction_val",
    "usage_total_180d", "usage_recent_90d", "usage_recent_30d",
    "usage_active_months_180d", "usage_trend_pct", "payment_to_usage_ratio"
]

print(f"📊 สรุปประชากรลูกค้าสำหรับวิเคราะห์ CLV (ณ วันที่ {CUTOFF_DATE}):")
print(f"   - ลูกค้าที่เข้าเกณฑ์ (Eligible)    : {len(clv_df):,} บัญชี")
print(f"   - ลูกค้าที่จ่ายเงินใน 6 เดือนข้างหน้า : {clv_df['target_pay_flag'].sum():,} บัญชี ({clv_df['target_pay_flag'].mean():.1%})")
print(f"   - ยอดเงินจริงที่เกิดขึ้นใน 6 เดือน  : ฿{clv_df['future_revenue_6m'].sum():,.2f}")
print(f"   - เฉลี่ยต่อเดือนทั้งพอร์ตโฟลิโอ      : ฿{clv_df['future_revenue_6m'].sum()/6:,.2f}/เดือน")

## ✂️ Step 4: แบ่งชุดข้อมูล Train (60%) / Validation (20%) / Test (20%)

In [ ]:
tr_df, hold_df = train_test_split(
    clv_df, test_size=0.40, random_state=42, stratify=clv_df["target_pay_flag"]
)
val_df, te_df = train_test_split(
    hold_df, test_size=0.50, random_state=42, stratify=hold_df["target_pay_flag"]
)

X_train, y_train_flag, y_train_rev = tr_df[feature_cols], tr_df["target_pay_flag"], tr_df["future_revenue_6m"]
X_val, y_val_flag, y_val_rev = val_df[feature_cols], val_df["target_pay_flag"], val_df["future_revenue_6m"]
X_test, y_test_flag, y_test_rev = te_df[feature_cols], te_df["target_pay_flag"], te_df["future_revenue_6m"]

print("✂️ แบ่งข้อมูลเรียบร้อย:")
print(f"   - Train Set : {len(X_train):,} บัญชี")
print(f"   - Val Set   : {len(X_val):,} บัญชี (ใช้ Calibrate ยอดรวมทั้งพอร์ต)")
print(f"   - Test Set  : {len(X_test):,} บัญชี (ประเมินความแม่นยำจริง)")

## 🧠 Step 5: เทรนโมเดล Two-Part Hurdle (LightGBM + CatBoost Log-Space)
1. **Classifier**: ทำนายโอกาสซื้อ `P(Pay > 0)` ด้วย LightGBM
2. **Regressor**: ทำนายมูลค่ายอดเงินด้วย CatBoost บน Log-Scale (`np.log1p`) เฉพาะผู้ที่จ่ายเงิน

In [ ]:
# 1. เทรน Classifier
pay_clf = LGBMClassifier(n_estimators=100, learning_rate=0.04, max_depth=5, random_state=42, verbose=-1)
pay_clf.fit(X_train, y_train_flag)

# 2. เทรน Regressor ใน Log-Space (เฉพาะคนที่จ่ายเงินจริงเพื่อเรียนรู้ขนาดแพ็กเกจ)
mask_tr_payers = y_train_rev > 0
amt_reg = CatBoostRegressor(iterations=250, learning_rate=0.04, depth=5, random_seed=42, verbose=0)
amt_reg.fit(X_train[mask_tr_payers], np.log1p(y_train_rev[mask_tr_payers]))

# 3. คำนวณยอดเงินบน Validation Set และหาตัวคูณปรับสเกล (Portfolio Multiplier)
val_probs = pay_clf.predict_proba(X_val)[:, 1]
val_amt = np.expm1(np.clip(amt_reg.predict(X_val), 0, 16))
val_expected_raw = val_probs * val_amt

portfolio_multiplier = float(y_val_rev.sum() / max(1e-6, val_expected_raw.sum()))
portfolio_multiplier = np.clip(portfolio_multiplier, 0.70, 1.40)  # ขอบเขตความปลอดภัย

print(f"✅ เทรน Two-Part Model สำเร็จ")
print(f"📐 Portfolio Magnitude Multiplier: {portfolio_multiplier:.4f}")

def predict_clv_6m(X_input):
    probs = pay_clf.predict_proba(X_input)[:, 1]
    raw_amt = np.expm1(np.clip(amt_reg.predict(X_input), 0, 16))
    clv_val = probs * raw_amt * portfolio_multiplier
    return probs, clv_val

## 📈 Step 6: ประเมินผลบนชุดทดสอบจริง (Test Set Evaluation)

In [ ]:
test_probs, test_clv_pred = predict_clv_6m(X_test)

actual_test_total = y_test_rev.sum()
pred_test_total = test_clv_pred.sum()
portfolio_bias = (pred_test_total - actual_test_total) / actual_test_total * 100

mask_te_payers = y_test_rev > 0
sp_payers, _ = spearmanr(y_test_rev[mask_te_payers], test_clv_pred[mask_te_payers])
sp_all, _ = spearmanr(y_test_rev, test_clv_pred)
mae_test = mean_absolute_error(y_test_rev, test_clv_pred)

print("🏆 ประสิทธิภาพของ Clean Hurdle CLV บน Test Set:")
print(f"   - ยอดเงินจริงรวม (Actual Total)       : ฿{actual_test_total:,.2f}")
print(f"   - ยอดเงินทำนายรวม (Predicted Total)   : ฿{pred_test_total:,.2f}")
print(f"   - Portfolio Bias                      : {portfolio_bias:+.2f}% (เป้าหมาย < ±10%)")
print(f"   - Spearman Ranking (เฉพาะผู้จ่ายเงิน) : {sp_payers:.4f} (เป้าหมาย > 0.60)")
print(f"   - Spearman Ranking (ลูกค้าทั้งหมด)    : {sp_all:.4f}")
print(f"   - MAE เฉลี่ยต่อราย                   : ฿{mae_test:,.2f}")

# วาดกราฟ Decile Revenue Capture
eval_df = pd.DataFrame({"actual": y_test_rev, "pred": test_clv_pred}).sort_values("pred", ascending=False)
eval_df["cum_actual"] = eval_df["actual"].cumsum() / actual_test_total * 100
eval_df["cum_pop"] = np.arange(1, len(eval_df) + 1) / len(eval_df) * 100

plt.figure(figsize=(9, 5))
plt.plot(eval_df["cum_pop"], eval_df["cum_actual"], color="#8e44ad", lw=2.5, label="Model Cumulative Revenue Capture")
plt.plot([0, 100], [0, 100], "k--", alpha=0.5, label="Random Benchmark")
plt.title("🎯 Cumulative Revenue Capture Curve on Test Set", fontsize=13, fontweight="bold")
plt.xlabel("% ประชากรลูกค้าที่ถูกจัดอันดับจากสูงสุดไปต่ำสุด")
plt.ylabel("% ยอดเงินรวมที่จับได้จริง (Cumulative Revenue %)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 🏷️ Step 7: การจัดกลุ่มมูลค่าลูกค้า (Value Tier Segmentation)

In [ ]:
# ทำนายลูกค้าทั้งหมดในพอร์ตโฟลิโอ
all_probs, all_clv = predict_clv_6m(clv_df[feature_cols])

def assign_value_tier(val):
    if val >= 100000:
        return "Whale (Enterprise)"
    elif val >= 10000:
        return "Core Tier"
    elif val >= 1000:
        return "Standard Tier"
    else:
        return "Low Tier"

value_tiers = [assign_value_tier(v) for v in all_clv]

output_df = pd.DataFrame({
    "acc_id": clv_df["acc_id"],
    "predicted_clv_6m": np.round(all_clv, 2),
    "pay_probability_6m": np.round(all_probs, 4),
    "value_tier": value_tiers,
    "total_revenue_all": clv_df["total_revenue_all"],
    "days_since_last_payment": clv_df["days_since_last_payment"]
})

output_path = "clv_predictions.csv"
output_df.to_csv(output_path, index=False)
print(f"🎉 บันทึกผลการทำนายเรียบร้อยที่: {output_path}")
print("\n📊 สรุปสัดส่วนตามกลุ่มมูลค่า (Value Tiers):")
print(output_df["value_tier"].value_counts())
print("\nตัวอย่างลูกค้าระดับ Whale 5 รายแรก:")
print(output_df[output_df['value_tier'] == 'Whale (Enterprise)'].sort_values('predicted_clv_6m', ascending=False).head(5))